# Retail Sales Forecasting: Exploratory Data Analysis & Feature Engineering
## Predicting Monthly Store Sales: From Raw Data to Model-Ready Dataset

### Objective
Explore retail transactions, customer behavior, and store features, then engineer a clean store-month dataset for machine learning.


## 1. Download Dataset (Google Colab)
Download all raw data files from Google Drive into the `raw_data` folder.


In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1kpdXXa3aBR-Zb6XcTLcOB8O0DdcHh8bv" -O raw_data


## 2. Imports & Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 3. Load the Raw Retail Data


In [ ]:
transactions = pd.read_csv('raw_data/TRANSACTIONS.csv', low_memory=False)
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
transactions['date'] = transactions['transaction_date'].dt.normalize()
transactions.head(3)


In [ ]:
stores = pd.read_csv('raw_data/STORES.csv')
calendar = pd.read_csv('raw_data/CALENDAR.csv')
calendar['date'] = pd.to_datetime(calendar['date'])
stores.head(3)


In [ ]:
promotions = pd.read_csv('raw_data/PROMOTIONS.csv')
store_promotions = pd.read_csv('raw_data/STORE_PROMOTIONS.csv')
store_promotions['start_date'] = pd.to_datetime(store_promotions['start_date'])
store_promotions['end_date'] = pd.to_datetime(store_promotions['end_date'])
promotions


In [ ]:
print("Transactions shape:    ", transactions.shape)
print("Stores shape:          ", stores.shape)
print("Calendar shape:        ", calendar.shape)
print("Promotions shape:      ", promotions.shape)
print("Store Promotions shape:", store_promotions.shape)


## 4. Data Integrity & Missing Values


In [ ]:
transactions.isnull().sum()


In [ ]:
stores.isnull().sum()


In [ ]:
(transactions['net_bill_value'] <= transactions['gross_bill_value']).mean() * 100


## 5. Store & Promotion Landscape


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=stores, x='store_format', hue='store_size_sqft', palette='Blues')
plt.title('Store Formats vs. Store Size (sqft)')
plt.xlabel('Store Format')
plt.ylabel('Store Count')
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=stores, x='location_type', hue='location_type', palette='crest', legend=False)
plt.title('Store Count by Location Type')
plt.xlabel('Location Type')
plt.ylabel('Store Count')
plt.show()


## 6. Daily Sales Trends


In [ ]:
daily_sales = transactions.groupby('date')['net_bill_value'].sum().reset_index()
daily_sales.head()


In [ ]:
plt.figure(figsize=(14, 4))
sns.lineplot(data=daily_sales, x='date', y='net_bill_value', color='teal')
plt.title('Daily Total Sales across All Stores')
plt.xlabel('Date')
plt.ylabel('Daily Total Net Sales (INR)')
plt.show()


## 7. Customer Shopping Behavior: Weekends & Festivals


In [ ]:
trx_cal = pd.merge(
    transactions,
    calendar[['date', 'is_weekend', 'is_festive_period']],
    on='date',
    how='left'
)


In [ ]:
daily_store = trx_cal.groupby(['store_id', 'date']).agg(
    total_sales=('net_bill_value', 'sum'),
    total_transactions=('transaction_id', 'count'),
    total_items=('total_items', 'sum'),
    is_weekend=('is_weekend', 'first'),
    is_festive_period=('is_festive_period', 'first')
).reset_index()

daily_store['ATV'] = daily_store['total_sales'] / daily_store['total_transactions']
daily_store['UPT'] = daily_store['total_items'] / daily_store['total_transactions']
daily_store['day_of_week'] = daily_store['date'].dt.day_name()
daily_store.head(3)


In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(9, 4))
sns.boxplot(data=daily_store, x='day_of_week', y='total_sales', order=day_order, hue='day_of_week', palette='Set2', legend=False)
plt.title('Daily Store Sales by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Sales (INR)')
plt.xticks(rotation=30)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=daily_store, x='is_weekend', y='ATV', hue='is_weekend', palette='pastel', legend=False)
plt.title('Average Transaction Value: Weekend vs Weekday')
plt.xticks([0, 1], ['Weekday', 'Weekend'])
plt.ylabel('ATV (INR)')
plt.show()


## 8. Festival Impact & Payment Mode Distribution


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=daily_store, x='is_festive_period', y='total_sales', hue='is_festive_period', palette='Blues', legend=False)
plt.title('Average Daily Sales: Non-Festive vs Festive Period')
plt.xticks([0, 1], ['Normal Days', 'Festive Period'])
plt.ylabel('Average Daily Sales (INR)')
plt.show()


In [ ]:
merged_txns = pd.merge(transactions, stores[['store_id', 'location_type']], on='store_id', how='left')

plt.figure(figsize=(8, 4))
sns.countplot(data=merged_txns, x='payment_mode', hue='location_type', palette='magma')
plt.title('Payment Mode Distribution by Location Type')
plt.xlabel('Payment Mode')
plt.ylabel('Transaction Count')
plt.show()


## 9. Feature Engineering: Monthly Aggregation
We aggregate transactions and merge store attributes, calendar indicators, and active promotion days to build `final_data.csv`.


In [ ]:
transactions['year'] = transactions['transaction_date'].dt.year
transactions['month'] = transactions['transaction_date'].dt.month

monthly_sales = transactions.groupby(['store_id', 'year', 'month']).agg(
    total_sales=('net_bill_value', 'sum'),
    total_transactions=('transaction_id', 'count'),
    total_items=('total_items', 'sum'),
    total_discount=('promo_discount_amount', 'sum')
).reset_index()

monthly_sales.head()


In [ ]:
df_monthly = pd.merge(monthly_sales, stores, on='store_id', how='left')
df_monthly.head(3)


In [ ]:
calendar['year'] = calendar['date'].dt.year
calendar['month'] = calendar['date'].dt.month

cal_monthly = calendar.groupby(['year', 'month']).agg(
    days_festive=('is_festive_period', 'sum'),
    days_weekend=('is_weekend', 'sum'),
    days_in_month=('date', 'count')
).reset_index()

df_monthly = pd.merge(df_monthly, cal_monthly, on=['year', 'month'], how='left')
df_monthly.head(3)


In [ ]:
sp_merged = pd.merge(store_promotions, promotions[['promo_id', 'promo_type']], on='promo_id', how='left')

promo_records = []
for _, row in sp_merged.iterrows():
    for day in pd.date_range(row['start_date'], row['end_date']):
        promo_records.append({
            'store_id': row['store_id'],
            'year': day.year,
            'month': day.month,
            'promo_type': row['promo_type'],
            'date': day
        })

promo_days_df = pd.DataFrame(promo_records).drop_duplicates(subset=['store_id', 'promo_type', 'date'])
promo_monthly = promo_days_df.groupby(['store_id', 'year', 'month', 'promo_type'])['date'].count().unstack(fill_value=0).reset_index()
promo_monthly = promo_monthly.rename(columns={col: f'days_{col}' for col in promo_monthly.columns if col not in ['store_id', 'year', 'month']})
promo_monthly.head(3)


In [ ]:
final_df = pd.merge(df_monthly, promo_monthly, on=['store_id', 'year', 'month'], how='left')

promo_cols = [c for c in final_df.columns if c.startswith('days_') and c not in ['days_festive', 'days_weekend', 'days_in_month']]
final_df[promo_cols] = final_df[promo_cols].fillna(0).astype(int)

final_df.to_csv('final_data.csv', index=False)
final_df.head(3)


## 10. Exploring the Monthly Dataset


In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(final_df['total_sales'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Monthly Store Sales')
plt.xlabel('Monthly Total Sales (INR)')
plt.ylabel('Store-Month Count')
plt.show()


In [ ]:
monthly_pattern = final_df.groupby('month')['total_sales'].mean().reset_index()

plt.figure(figsize=(9, 4))
sns.barplot(data=monthly_pattern, x='month', y='total_sales', hue='month', palette='Blues_d', legend=False)
plt.title('Average Monthly Sales Across All Stores')
plt.xlabel('Month')
plt.ylabel('Average Sales (INR)')
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=final_df, x='month', y='total_sales', hue='year', estimator='mean', marker='o', palette='tab10')
plt.title('Year-over-Year Sales Trends')
plt.xticks(range(1, 13))
plt.xlabel('Month')
plt.ylabel('Average Sales (INR)')
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=final_df, x='store_format', y='total_sales', hue='store_format', palette='Set2', legend=False)
plt.title('Monthly Sales by Store Format')
plt.xlabel('Store Format')
plt.ylabel('Monthly Sales (INR)')
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=final_df, x='store_size_sqft', y='total_sales', hue='store_format', alpha=0.7)
plt.title('Store Size (sqft) vs Monthly Sales')
plt.xlabel('Store Size (sqft)')
plt.ylabel('Monthly Sales (INR)')
plt.show()


In [ ]:
corr_features = [
    'total_sales', 'store_size_sqft', 'competition_density',
    'avg_monthly_footfall', 'days_festive', 'days_weekend'
] + promo_cols

corr_mat = final_df[corr_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_mat, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Matrix: Monthly Features vs Total Sales')
plt.show()
